# Text Classification and Quantitative Evaluation
#### By Jeremy Merrill for Lede 2025


This notebook extracts podcast guests from episode descriptions using the OpenAI interface.

It includes a cost estimation and code for quantitative evaluation (seeing how you did, then adjusting to try to do better).

It can serve as a template for any kind of extraction, and you should feel free to copy and reuse it.

**First thing to do: Make a copy. File -> Save a copy in Drive OR Download -> Download .ipynb (and then run this on your computer)**


TODO:
 - make sure that the notebook really does let you re-run the prompt on the same sample (and check the results without re-coding them)


In [ ]:
## installing relevant packages
!pip install --quiet openai strip_tags tiktoken

import os
import pandas as pd
from openai import OpenAI
from IPython.display import display, Markdown
from tqdm.auto import tqdm # makes pretty progress bars
tqdm.pandas()

IN_COLAB = True # make this False if you're on your laptop

if IN_COLAB:
  from google.colab import files
else:
  from dotenv import load_dotenv # you'll want this if you're running on your laptop

In [ ]:
## setting up API key

#  # uncomment if you're running on your laptop in a datakit
# if you're in Colab, put the API key in the Secrets (the key icon on the far left of the screen.)
# the "Name" needs to be exactly `API_KEY` and you need to make sure that the Notebook Access checkbox is checked.
if IN_COLAB:
  from google.colab import userdata
  api_key = userdata.get('OPENAI_API_KEY_LEDE')
else:
  load_dotenv()
  api_key = os.environ['OPENAI_API_KEY_LEDE']

## importing data
if IN_COLAB:
  data_dir = ""
  # see the Podcast data https://docs.google.com/spreadsheets/d/1GjVHOrDbBBimoVkToXTailtPRuZs8KALGiWj--RLktI/edit?gid=1305112055#gid=1305112055
  ## if you're reusing this notebook, replace this with a link to your data
  ## you will probably have to replace both the 1GjV thing and 1305112055 with the numbers from your Google Sheet
  data_raw = pd.read_csv('https://docs.google.com/spreadsheets/d/1GjVHOrDbBBimoVkToXTailtPRuZs8KALGiWj--RLktI/export?format=csv&gid=1305112055')
else:
  ## or, use a CSV replace this with your data's filename
  data_dir = "data/raw/"
  data_raw = pd.read_csv("podcast_summaries1000.csv")


SIZE_OF_SAMPLE = 50
##
## take a random sample to test our prompts against
text_column_name = "summary_clean" # change this to match your own dataset

data_raw.dropna(subset=[text_column_name], inplace=True)
data_raw.set_index("platform_item_id", inplace=True)
data_raw["created_at"] = pd.to_datetime(data_raw["created_at"])

data_full = data_raw

data_sample = data_full.sample(n=SIZE_OF_SAMPLE)

In [ ]:
data_sample.head()

,source_name,created_at,channel_name,summary,summary_clean,tokens_gpt41mini,guests_latest
platform_item_id,,,,,,,
bfb03cf1-89e6-4178-9254-b2840145c2ae,Clay Travis and Buck Sexton,2025-02-14 20:20:00,The Clay Travis and Buck Sexton Show,Trump's coalition is a force of super-geniuses...,Trump's coalition is a force of super-geniuses...,94,[]
https://www.americaoutloud.news/?p=193847,AMERICA OUT LOUD PODCAST NETWORK,2025-04-05 13:22:16,AMERICA OUT LOUD PODCAST NETWORK,Looking 4 Healing Radio with Justin Feldman – ...,Looking 4 Healing Radio with Justin Feldman – ...,77,[]
81f05681-b1b2-40cc-a1e9-6d204ac0c437,Kyle Seraphin,2025-02-14 16:10:29,The Kyle Seraphin Show,<p>The Kyle Seraphin Show is streamed live at ...,The Kyle Seraphin Show is streamed live at 9:3...,119,['SmoosieQ']
e60e6e70-5441-11ef-9c32-67536f3074fe,2 Bears 1 Cave,2025-04-07 12:00:00,"2 Bears, 1 Cave with Tom Segura & Bert Kreischer","\n <p>Check out Bert's new special ""Luc...","Check out Bert's new special ""Lucky"" streaming...",542,['Kyle Dunnigan']
df06667a-e96d-11ef-ac4c-874206e388df,Daniel Horowitz,2025-02-12 18:19:00,Conservative Review with Daniel Horowitz,\n <p>So many of us have been inspired ...,So many of us have been inspired by Trump’s ex...,170,['Ryan Walters']


In [ ]:

## setting up our connection to OpenAI client
client = OpenAI(api_key=api_key)

A basic format for a prompt is below. Using the web browser of chat GPT, tweak this until you're pretty consistently getting reasonable results.

In [ ]:
## prompt
## start with the one YOU tried in the LLM web interface
## don't worry, we'll adjust it later.

# Be sure to list the categories with the magic phrase {categories}
prompt_base = """

YOUR PROMPT GOES HERE. {podcast_name} {podcast_description}

"""

# for convenience, make a column in our dataframes with the prompt for each item
data_sample_prompt_column = data_sample.apply(lambda row: prompt_base.format(
    podcast_name=row["source_name"],
    podcast_description=row[text_column_name]
), axis="columns")
full_data_prompt_column = data_full.apply(lambda row: prompt_base.format(
    podcast_name=row["source_name"],
    podcast_description=row[text_column_name]
), axis="columns")

In [ ]:
## cost estimation

from strip_tags import strip_tags
import tiktoken
def count_tokens(model, text):
    encoding = tiktoken.encoding_for_model(model if model != 'gpt-4.1-mini' else 'gpt-4o')
    tokens = encoding.encode(text)
    return len(tokens)

input_token_costs = {
    "gpt-4.1-mini": 0.4 / 1_000_000,  # https://openai.com/api/pricing/
    "mistral-medium-latest": 0.4 / 1_000_000, # https://mistral.ai/pricing#api-pricing
    "mistral-large-latest": 2.0 / 1_000_000
}
def estimate_cost(model, token_count):
    return token_count * input_token_costs[model]

MODEL_TO_USE = 'gpt-4.1-mini'

count_tokens_for_our_model = lambda text: count_tokens(MODEL_TO_USE, text)

The code blocks below will let you estimate costs. If it's too pricey with `gpt-4o`, consider a cheaper model.

In [ ]:
## cost estimation sample
token_count_sample = count_tokens_for_our_model("SAMPLE RESPONSE SAMPLE RESPONSE".join(data_sample_prompt_column))
"Sample would cost: ${:.2f}".format(estimate_cost(MODEL_TO_USE, token_count_sample))

'Sample would cost: $0.00'

In [ ]:
## cost estimation for full dataset
token_count_full = count_tokens_for_our_model("SAMPLE RESPONSE SAMPLE RESPONSE".join(full_data_prompt_column))
"Full dataset would cost: ${:.2f}".format(estimate_cost(MODEL_TO_USE, token_count_full))

'Full dataset would cost: $0.08'

Below is the actual call to the OpenAI AI. Run your sample on the same model you want to use for your real data.

**BUT FIRST**: adjust the categories below to match what you defined in your prompt.

In [ ]:
## function to actually send the prompt to OpenAI and get the answer
from pydantic import BaseModel
from typing import List


# this is a fancy way of forcing OpenAI to give us a list.
class PodcastExtractionListOfGuests(BaseModel):
    guests: List[str]

def classify(prompt_including_tweet):

    # put our prompt into the blob that OpenAI expects
    messages = [
        {
            "role": "system",
            "content": "You are a helpful assistant.",
        },
        {
            "role": "user",
            "content": prompt_including_tweet,
        }
    ]

    chat_completion = client.responses.parse(
        input=messages,
        model='gpt-4o',
        text_format=PodcastExtractionListOfGuests,
    )
    # get the answer out of the blob that OpenAI returns.
    resp = chat_completion.output_parsed.guests if chat_completion.output_parsed else []
    return resp

In [ ]:
# ask ChatGPT about each and every tweet IN THE SAMPLE, using the prompt we made above

data_sample["ai_guess"] = data_sample_prompt_column.progress_apply(classify)

  0%|          | 0/50 [00:00<?, ?it/s]

In [ ]:
display(Markdown("## Here are a few results. How good did we do?"))
with pd.option_context("display.max_colwidth", 500):
  display(
      data_sample[["source_name", text_column_name, "ai_guess"]].head(10)
  )

## Here are a few results. How good did we do?

,source_name,summary_clean,ai_guess
platform_item_id,,,
bfb03cf1-89e6-4178-9254-b2840145c2ae,Clay Travis and Buck Sexton,"Trump's coalition is a force of super-geniuses'. NYC Mayor Eric Adams and Border Czar Tom Homan together on Fox & Friends. Auditing the IRS, it's about time. CNN has to admit that Clay has a point when it comes to Democrat men. Who is the most masculine Democrat?Follow Clay & Buck on YouTube: https://www.youtube.com/c/clayandbuckSee omnystudio.com/listener for privacy information.","[NYC Mayor Eric Adams, Border Czar Tom Homan]"
https://www.americaoutloud.news/?p=193847,AMERICA OUT LOUD PODCAST NETWORK,"Looking 4 Healing Radio with Justin Feldman – The truth is, transforming your health isn’t about quick fixes or willpower — it’s about having a clear, structured approach that aligns with your values, your lifestyle, and your deeper purpose. In this episode of Looking 4 Healing Radio, we break down The Simple Step-by-Step Blueprint to Transform Your Health and Life...",[Justin Feldman]
81f05681-b1b2-40cc-a1e9-6d204ac0c437,Kyle Seraphin,"The Kyle Seraphin Show is streamed live at 9:30a ET / 8:30a CT.Special Guest: https://x.com/SmoosieQJoin the best LIVE chat on the web on Rumble, orfind me on Spotify: https://KyleSeraphinShow.com__________________________________________________Our Sponsors make this program possible:https://PatriotCoolers.com/collections/kyle-seraphin (Tumblers & Coolers)ANDhttp://patriot-protect.com/KYLE (15% off Protecting yourself from scams/Identity theft)",[SmoosieQ]
e60e6e70-5441-11ef-9c32-67536f3074fe,2 Bears 1 Cave,"Check out Bert's new special ""Lucky"" streaming on Netflix! Tom just announced Fall dates for his Come Together Tour at https://tomsegura.com/tour. Presale starts April 2nd, use code TOMMY.Watch Kyle Dunnigan on Kill Tony on Netflix today!SPONSORS:- You can find Cremo’s new line of antiperspirants and deodorants at Target or https://Target.com- Sign up for a $1 per month trial period at https://shopify.com/bears.- Brought to you by BetterHelp. Visit https://betterhelp.com/bears to get 10% off...","[Tom Segura, Kyle Dunnigan]"
df06667a-e96d-11ef-ac4c-874206e388df,Daniel Horowitz,"So many of us have been inspired by Trump’s executive orders, but unfortunately most of them will not last, because Republicans have not changed who they are. I explain why Trump needs to push back against judicial supremacism and demand a better budget reconciliation bill in Congress in order for him to stick the landing on his agenda. Sadly, Republicans have not changed even in deep-red states. We’re joined today by Ryan Walters, the Oklahoma state superintendent of public instruction, who...","[Ryan Walters, Oklahoma State Superintendent of Public Instruction]"
253ccd52-801c-11ef-9cdc-9f70484ba0df,Pivot (Kara Swisher and Scott Galloway),"Kara and Scott discuss President Trump’s call with Vladimir Putin about Ukraine, inflation rising 3% in January, and Elon Musk’s Oval Office appearance. Then, Sam Altman has some harsh thoughts about Elon Musk following his bid for OpenAI, an AI deal between China’s BYD and DeepSeek threatens Tesla, and the AP gets punished for deadnaming the “Gulf of America.” Stick around for Scott’s prediction on Tesla.Follow us on Instagram and Threads at @pivotpodcastofficial.Follow us on Bluesky at @pi...",[]
a8c2a7e9-a52e-40a9-824f-b26d013f5ab5,Clay Travis and Buck Sexton,"Slew of EOs, including only American flags at U.S. embassies. Huge decline in illegal border crossings. NYC Mayor Adams reveals a truth. Fetterman rumors. Prayer service lecture asserted a false premise and did more damage than good. Boob-gate!Follow Clay & Buck on YouTube: https://www.youtube.com/c/clayandbuckSee omnystudio.com/listener for privacy information.",[]
839ddf6e-ba51-4185-b005-825947e855cb,Joe Oltmann,"Joe is untamed and joined by his soul brother, Orlando Owens, today! Joe and Orlando begin the show with a conversation on Move The Needle and its mission to educate voters in Milwaukee ab

## Your turn!

Now it's your turn to do the whole workflow we did for classification, but for this extraction task.

1. Go hand-code some answers in Google Docs/Excel (or another tool)
2. Do some quantitative evaluation of the results. This will be a little trickier for extraction. First, you'll have to deal with correct answers that are spelled a little differently, like `Ryan Walters, Oklahoma State Superintendent of Public Instruction` versus `Ryan Walters. Second, you'll have to figure out how to deal with partial answers, like if the right answer is `[Tom Segura, Kyle Dunnigan]` but the model only got `[Tom Segura]`. You can figure it out!
3. Then, improve your prompt.
4. Finally, do extraction over the whole dataset.

The code from the extraction notebook will be your friend, as will coding assistants (including Gemini here in Colab, but any of them will do.)

In [ ]:
JEREMY_PROMPT = """
You're a helpful research assistant.
Return a JSON list of the names all the guests appearing on the podcast whose summary is appended.
Do not include the names of people who are merely discussed on the podcast unless there's an explicit indication that they are a guest who appears on the podcast.
Do not include the name of the person who hosts the podcast, if that person's name is included in the podcast's name. Do include guest hosts.
If you are unsure if someone appears as a guest on a podcast or is merely discussed, do not include them in the response.
Do not provide any English text, only a list; do not include the titles of the guests. If there are no guests, return just an empty list.
JSON List Schema: ["Person 1", "Person 2"]
Review your analysis carefully before returning a response.
Examples:
Podcast Name: "Always Be Pirate Shipping with John Smith"
Podcast Summary: "Our guest today is Texas Attorney General Jeremy Merrill, and we talk about computers."
Response: ["Jeremy Merrill"]
Podcast Name: "Always Be Pirate Shipping with John Smith"
Podcast Summary: "Our guest today is Texas Attorney General Jeremy Merrill, and we talk about Joe Biden's bizarre claims about computers and Britney Spear's interview with Nikita Khrushchev."
Response: ["Jeremy Merrill"]
Podcast Name: "Always Be Pirate Shipping with Jeremy Merrill"
Podcast Summary: "On this podcast, Jeremy does a monologue."
Response: []
Podcast Name: "Internet Garbage Tech Pop Culture"
Podcast Summary: "Jane Doe explains twelve things. Jake Kara says Democrats are moving to New York because rent is cheaper now."
Response: []
Podcast Name: "Always Be Pirate Shipping with Jeremy Merrill"
Podcast Summary: "Today, Dr. Bob Dylan joins to discuss AI prompting, and second, we talk about CloudFormation templates with Rev. Jane Doe. Thanks for subscribing to the Always Be Pirate Shipping with Jeremy Merrill podcast."
Response: ["Bob Dylan", "Jane Doe"]
Podcast Name: {channel_name}
Podcast Summary: "{summary_clean}"
Response:
        """